In [ ]:
import pandas as pd
import numpy as np
from shapely.geometry import Point, Polygon

# === Constants ===
dtw_lat, dtw_lon = 42.2125, -83.3534
radius_deg = 3

# === STAR Fixes & TRACON Polygon ===
star_fixes = {
    "BONZZ": (-82.7972, 41.7483), "CRAKN": (-82.9405, 41.6730), "CUUGR": (-83.0975, 42.3643),
    "FERRL": (-82.6093, 42.4165), "GRAYT": (-83.6020, 42.9150), "HANBL": (-84.1773, 41.7375),
    "HAYLL": (-84.2975, 41.9662), "HTROD": (-83.3442, 42.0278), "KKISS": (-83.7620, 42.5443),
    "KLYNK": (-82.9888, 41.8793), "LAYKS": (-83.5498, 42.8532), "LECTR": (-84.0217, 41.9183),
    "RKCTY": (-83.9603, 42.6869), "VCTRZ": (-84.0670, 41.9878)
}
ordered_fix_names = ["CRAKN", "BONZZ", "FERRL", "LAYKS", "GRAYT", "RKCTY", "HAYLL", "HANBL"]
tracon_polygon = Polygon([star_fixes[name] for name in ordered_fix_names])

# === Pre-TRACON Circle Polygon ===
theta = np.linspace(0, 2 * np.pi, 400)
circle_coords = [(dtw_lon + radius_deg * np.cos(t), dtw_lat + radius_deg * np.sin(t)) for t in theta]
circle_polygon = Polygon(circle_coords)

# === Load and preprocess data ===
df = pd.read_csv("filtered_rows.csv", low_memory=False)
df['recTime'] = pd.to_datetime(df['recTime'])

# === Get first 10 unique acId ===
top_acids = df['acId'].dropna().unique()[:1200]

# === Store results ===
records = []

for ac in top_acids:
    sub_df = df[df['acId'] == ac].sort_values('recTime')
    enter_circle_time = None
    enter_tracon_time = None
    enter_circle_coord = None
    enter_tracon_coord = None

    for _, row in sub_df.iterrows():
        lon, lat = row['coord2'], row['coord1']
        p = Point(lon, lat)

        if enter_circle_time is None and circle_polygon.covers(p):
            enter_circle_time = row['recTime']
            enter_circle_coord = (lon, lat)

        if enter_circle_time is not None and enter_tracon_time is None and tracon_polygon.covers(p):
            enter_tracon_time = row['recTime']
            enter_tracon_coord = (lon, lat)
            break  
    if enter_circle_time and enter_tracon_time:
        duration = (enter_tracon_time - enter_circle_time).total_seconds()
        records.append({
            'acId': ac,
            'enter_pre_tracon': enter_circle_time,
            'enter_tracon': enter_tracon_time,
            'duration_seconds': duration,
            'enter_pre_tracon_coord': enter_circle_coord,
            'enter_tracon_coord': enter_tracon_coord
        })

# === Output as DataFrame ===
result_df = pd.DataFrame(records)
print(result_df)

result_df.to_csv("flight_entry_times_with_coords.csv", index=False)

In [ ]:
import pandas as pd
from shapely.geometry import Point, Polygon
import numpy as np

# Load data
df = pd.read_csv("filtered_rows.csv", low_memory=False)
df['recTime'] = pd.to_datetime(df['recTime'])

# === Constants ===
dtw_lat, dtw_lon = 42.2125, -83.3534
radius_deg = 3

# === STAR Fixes & TRACON Polygon ===
star_fixes = {
    "BONZZ": (-82.7972, 41.7483), "CRAKN": (-82.9405, 41.6730), "CUUGR": (-83.0975, 42.3643),
    "FERRL": (-82.6093, 42.4165), "GRAYT": (-83.6020, 42.9150), "HANBL": (-84.1773, 41.7375),
    "HAYLL": (-84.2975, 41.9662), "HTROD": (-83.3442, 42.0278), "KKISS": (-83.7620, 42.5443),
    "KLYNK": (-82.9888, 41.8793), "LAYKS": (-83.5498, 42.8532), "LECTR": (-84.0217, 41.9183),
    "RKCTY": (-83.9603, 42.6869), "VCTRZ": (-84.0670, 41.9878)
}
ordered_fix_names = ["CRAKN", "BONZZ", "FERRL", "LAYKS", "GRAYT", "RKCTY", "HAYLL", "HANBL"]
tracon_polygon = Polygon([star_fixes[name] for name in ordered_fix_names])

# === Pre-TRACON Circle Polygon ===
theta = np.linspace(0, 2 * np.pi, 400)
circle_coords = [(dtw_lon + radius_deg * np.cos(t), dtw_lat + radius_deg * np.sin(t)) for t in theta]
circle_polygon = Polygon(circle_coords)

# === Filter for specific aircraft ===
ac = "DAL1296_KDENtoKDTW"
sub_df = df[df['acId'] == ac].sort_values('recTime')

enter_circle_time = None
enter_tracon_time = None
leave_tracon_time = None
enter_circle_coord = None
enter_tracon_coord = None
leave_tracon_coord = None

for _, row in sub_df.iterrows():
    lon, lat = row['coord2'], row['coord1']
    p = Point(lon, lat)

    if enter_circle_time is None and circle_polygon.covers(p):
        enter_circle_time = row['recTime']
        enter_circle_coord = (lon, lat)

    if enter_circle_time is not None and enter_tracon_time is None and tracon_polygon.covers(p):
        enter_tracon_time = row['recTime']
        enter_tracon_coord = (lon, lat)

    if enter_tracon_time is not None and not tracon_polygon.covers(p):
        leave_tracon_time = row['recTime']
        leave_tracon_coord = (lon, lat)
        break

# Prepare result
result = {
    'acId': ac,
    'enter_pre_tracon': enter_circle_time,
    'enter_tracon': enter_tracon_time,
    'leave_tracon': leave_tracon_time,
    'enter_pre_tracon_coord': enter_circle_coord,
    'enter_tracon_coord': enter_tracon_coord,
    'leave_tracon_coord': leave_tracon_coord
}
result